# Create baseline with logistic regression and compare to other models

In [33]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

## Import data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/education-ml-research/ASSISTments2009/skill_builder_data.csv"
df = pd.read_csv(path, encoding='latin1')

print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.head())

Mounted at /content/drive


/tmp/ipykernel_1061/1480988791.py:5: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, encoding='latin1')


(525534, 30)
Index(['order_id', 'assignment_id', 'user_id', 'assistment_id', 'problem_id',
       'original', 'correct', 'attempt_count', 'ms_first_response',
       'tutor_mode', 'answer_type', 'sequence_id', 'student_class_id',
       'position', 'type', 'base_sequence_id', 'skill_id', 'skill_name',
       'teacher_id', 'school_id', 'hint_count', 'hint_total', 'overlap_time',
       'template_id', 'answer_id', 'answer_text', 'first_action',
       'bottom_hint', 'opportunity', 'opportunity_original'],
      dtype='object')
order_id                  int64
assignment_id             int64
user_id                   int64
assistment_id             int64
problem_id                int64
original                  int64
correct                   int64
attempt_count             int64
ms_first_response         int64
tutor_mode               object
answer_type              object
sequence_id               int64
student_class_id          int64
position                  int64
type                 

## Remove rows with missing `skill_id`

In [34]:
# Remove attempts with missing skill IDs
df = df.dropna(subset=["skill_id"]).copy()

print("Dataset shape after removing missing skill IDs:", df.shape)
print("Missing skill IDs:", df["skill_id"].isna().sum())

Dataset shape after removing missing skill IDs: (459208, 37)
Missing skill IDs: 0


## Sort chronologically

In [35]:
df = df.sort_values(
    ["user_id", "order_id"]
).reset_index(drop=True)

print(df[["user_id", "order_id", "skill_id", "correct"]].head(20))

    user_id  order_id  skill_id  correct
0        14  21617623       2.0        0
1        14  21617623      37.0        0
2        14  21617623      70.0        0
3        14  21617632       2.0        1
4        14  21617632      37.0        1
5        14  21617632      70.0        1
6        14  21617641       2.0        0
7        14  21617641      37.0        0
8        14  21617641      70.0        0
9        14  21617650       2.0        0
10       14  21617650      37.0        0
11       14  21617650      70.0        0
12       14  21617659       2.0        0
13       14  21617659      37.0        0
14       14  21617659      70.0        0
15       14  21617667       2.0        0
16       14  21617667      37.0        0
17       14  21617667      70.0        0
18       14  21617675       2.0        0
19       14  21617675      37.0        0


## Create student accuracy feature

In [36]:
df["student_previous_attempts"] = (
    df.groupby("user_id").cumcount()
)

df["student_previous_correct"] = (
    df.groupby("user_id")["correct"]
    .cumsum()
    .shift(1)
)

df["student_accuracy"] = (
    df["student_previous_correct"] /
    df["student_previous_attempts"].replace(0, np.nan)
)

## If there is no prior data, fill 0.5 for accuracy
df["student_accuracy"] = df["student_accuracy"].fillna(0.5)



## Create skill accuracy feature

In [37]:
df["skill_attempts"] = (
    df.groupby(["user_id", "skill_id"])
    .cumcount()
)

df["skill_previous_correct"] = (
    df.groupby(["user_id", "skill_id"])["correct"]
    .cumsum()
    .shift(1)
)

df["skill_accuracy"] = (
    df["skill_previous_correct"] /
    df["skill_attempts"].replace(0, np.nan)
)

## If there is no prior data, fill 0.5 for accuracy
df["skill_accuracy"] = df["skill_accuracy"].fillna(0.5)

## Create time since last attempt feature

In [38]:
df["time_since_last"] = (
    df.groupby("user_id")["order_id"]
      .diff()
      .fillna(0)
)

## Check features

In [39]:
print(
    df[
        [
            "user_id",
            "skill_id",
            "order_id",
            "student_accuracy",
            "skill_accuracy",
            "skill_attempts",
            "time_since_last",
            "correct"
        ]
    ].head(20)
)

## There should be 0 missing values
print(df[
    [
        "student_accuracy",
        "skill_accuracy",
        "skill_attempts",
        "time_since_last"
    ]
].isna().sum())

    user_id  skill_id  order_id  student_accuracy  skill_accuracy  \
0        14       2.0  21617623          0.500000        0.500000   
1        14      37.0  21617623          0.000000        0.500000   
2        14      70.0  21617623          0.000000        0.500000   
3        14       2.0  21617632          0.000000        0.000000   
4        14      37.0  21617632          0.250000        1.000000   
5        14      70.0  21617632          0.400000        1.000000   
6        14       2.0  21617641          0.500000        0.500000   
7        14      37.0  21617641          0.428571        0.500000   
8        14      70.0  21617641          0.375000        0.500000   
9        14       2.0  21617650          0.333333        0.333333   
10       14      37.0  21617650          0.300000        0.333333   
11       14      70.0  21617650          0.272727        0.333333   
12       14       2.0  21617659          0.250000        0.250000   
13       14      37.0  21617659   

## Create train and test data


In [40]:
from sklearn.model_selection import train_test_split

students = df["user_id"].unique()

## 80/20 split
train_students, test_students = train_test_split(
    students,
    test_size=0.20,
    random_state=42
)

print("Train students:", len(train_students))
print("Test students:", len(test_students))

lr_train = df[df["user_id"].isin(train_students)].copy()
lr_test = df[df["user_id"].isin(test_students)].copy()

print("Train shape:", lr_train.shape)
print("Test shape:", lr_test.shape)

Train students: 3330
Test students: 833
Train shape: (361682, 37)
Test shape: (97526, 37)


## Define features

In [41]:
feature_columns = [
    "student_accuracy",
    "skill_accuracy",
    "skill_attempts",
    "time_since_last"
]

X_train = lr_train[feature_columns]
y_train = lr_train["correct"]

X_test = lr_test[feature_columns]
y_test = lr_test["correct"]

## Train the model

In [42]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

lr_model.fit(X_train, y_train)

print("Logistic regression training complete.")

Logistic regression training complete.


## Calculate AUC

In [43]:
from sklearn.metrics import roc_auc_score

test_predictions = lr_model.predict_proba(X_test)[:, 1]

lr_auc = roc_auc_score(
    y_test,
    test_predictions
)

print(f"Logistic Regression Test AUC: {lr_auc:.6f}")

Logistic Regression Test AUC: 0.721214


## Logistic Regression Baseline

A logistic regression baseline was implemented using four features:

- Student overall accuracy
- Skill accuracy
- Number of previous attempts on the skill
- Time since the student's previous attempt

Attempts with missing skill IDs were excluded. The model was trained using the same student-level train/test split as the other knowledge tracing models.

**Test AUC: 0.721214**

## Model Comparison — ASSISTments 2009

| Model | Dataset | Key Settings | Test AUC |
|:---|:---|:---|---:|
| **SAKT** | ASSISTments 2009 (80/20 split) | Sequence length = 100, 5 attention heads, dropout = 0.2, batch size = 128, 100 epochs, Adam, best LR = 0.0001, d = 125 | **0.809785** |
| **BKT** | ASSISTments 2009 (80/20 split) | 123 skills, `num_fits=1`, C++ backend, parallel training | **0.794503** |
| **Logistic Regression** | ASSISTments 2009 (80/20 split) | Student accuracy, skill accuracy, previous skill attempts, time since last attempt | **0.721214** |
